In [0]:
-- 步骤1: 创建临时视图读取Bronze数据
CREATE OR REPLACE VIEW bronze_order_info_raw AS
SELECT 
    *,
    _metadata.file_path as source_file,
    current_timestamp() as load_timestamp
FROM read_files('abfss://bronze@sahyivy.dfs.core.windows.net/gmall/order_info/2026-01-26/15/*.parquet');



In [0]:
%sql
select * from bronze_order_info_raw

In [0]:
-- 步骤2: 保存到Silver层（Delta格式）
drop table if exists silver_orders_info;

CREATE TABLE IF NOT EXISTS silver_orders_info
(
    id                     BIGINT,
    consignee              STRING,
    consignee_tel          STRING,
    total_amount           DECIMAL(10,2),
    order_status           STRING,
    user_id                BIGINT,
    payment_way            STRING,
    delivery_address       STRING,
    order_comment          STRING,
    out_trade_no           STRING,
    trade_body             STRING,
    create_time            TIMESTAMP,
    operate_time           TIMESTAMP,
    expire_time            TIMESTAMP,
    process_status         STRING,
    tracking_no            STRING,
    parent_order_id        BIGINT,
    img_url                STRING,
    province_id            INT,
    activity_reduce_amount DECIMAL(16,2),
    coupon_reduce_amount   DECIMAL(16,2),
    original_total_amount  DECIMAL(16,2),
    feight_fee             DECIMAL(16,2),
    feight_fee_reduce      DECIMAL(16,2),
    refundable_time        TIMESTAMP,
    source_file            STRING,
    load_timestamp         TIMESTAMP,
    update_timestamp       TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/silver/gmall/order_info'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.enableChangeDataFeed' = 'true'
);

In [0]:
select * from silver_orders_info

In [0]:
-- 步骤3: 合并新数据到Silver表（UPSERT）
MERGE INTO silver_orders_info AS target
USING bronze_order_info_raw AS source
ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET                
                target.consignee              =source.consignee             ,
                target.consignee_tel          =source.consignee_tel         ,
                target.total_amount           =source.total_amount          ,
                target.order_status           =source.order_status          ,
                target.user_id                =source.user_id               ,
                target.payment_way            =source.payment_way           ,
                target.delivery_address       =source.delivery_address      ,
                target.order_comment          =source.order_comment         ,
                target.out_trade_no           =source.out_trade_no          ,
                target.trade_body             =source.trade_body            ,
                target.create_time            =source.create_time           ,
                target.operate_time           =source.operate_time          ,
                target.expire_time            =source.expire_time           ,
                target.process_status         =source.process_status        ,
                target.tracking_no            =source.tracking_no           ,
                target.parent_order_id        =source.parent_order_id       ,
                target.img_url                =source.img_url               ,
                target.province_id            =source.province_id           ,
                target.activity_reduce_amount =source.activity_reduce_amount,
                target.coupon_reduce_amount   =source.coupon_reduce_amount  ,
                target.original_total_amount  =source.original_total_amount ,
                target.feight_fee             =source.feight_fee            ,
                target.feight_fee_reduce      =source.feight_fee_reduce     ,
                target.refundable_time        =source.refundable_time       ,
                target.source_file            =source.source_file           ,
                target.load_timestamp         =source.load_timestamp        ,
                target.update_timestamp = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (id                    ,
                consignee             ,
                consignee_tel         ,
                total_amount          ,
                order_status          ,
                user_id               ,
                payment_way           ,
                delivery_address      ,
                order_comment         ,
                out_trade_no          ,
                trade_body            ,
                create_time           ,
                operate_time          ,
                expire_time           ,
                process_status        ,
                tracking_no           ,
                parent_order_id       ,
                img_url               ,
                province_id           ,
                activity_reduce_amount,
                coupon_reduce_amount  ,
                original_total_amount ,
                feight_fee            ,
                feight_fee_reduce     ,
                refundable_time       ,
                source_file           ,
                load_timestamp        ,
                update_timestamp
                )
        VALUES (source.id                    ,
                source.consignee             ,
                source.consignee_tel         ,
                source.total_amount          ,
                source.order_status          ,
                source.user_id               ,
                source.payment_way           ,
                source.delivery_address      ,
                source.order_comment         ,
                source.out_trade_no          ,
                source.trade_body            ,
                source.create_time           ,
                source.operate_time          ,
                source.expire_time           ,
                source.process_status        ,
                source.tracking_no           ,
                source.parent_order_id       ,
                source.img_url               ,
                source.province_id           ,
                source.activity_reduce_amount,
                source.coupon_reduce_amount  ,
                source.original_total_amount ,
                source.feight_fee            ,
                source.feight_fee_reduce     ,
                source.refundable_time       ,
                source.source_file           , 
                source.load_timestamp        ,
                current_timestamp()
                );

In [0]:
-- 创建日期维度表
CREATE TABLE IF NOT EXISTS gold_dim_date
USING DELTA
LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/dim_date'
AS
WITH date_series AS (
    SELECT explode(sequence(
        to_date('2020-01-01'), 
        to_date('2030-12-31'), 
        interval 1 day
    )) as date
)
SELECT 
    -- 代理键
    CAST(date_format(date, 'yyyyMMdd') AS INT) as date_key,
    
    -- 日期属性
    date as full_date,
    YEAR(date) as year,
    QUARTER(date) as quarter,
    MONTH(date) as month,
    DAY(date) as day,
    
    -- 周信息
    WEEKOFYEAR(date) as week_of_year,
    DAYOFWEEK(date) as day_of_week,
    
    -- 业务标记
    CASE WHEN DAYOFWEEK(date) IN (1,7) THEN 1 ELSE 0 END as is_weekend,
    CASE WHEN MONTH(date) = 12 AND DAY(date) = 25 THEN 1 ELSE 0 END as is_christmas
    
FROM date_series;

In [0]:
select * from gold_dim_date

In [0]:
drop view bronze_order_info_raw